In [5]:
import requests
from tqdm import tqdm
import time
import os
from dotenv import load_dotenv
import mwparserfromhell
import re
import json
from datetime import datetime

# Data Fetching for movie plots

### Use TMDB API to fetch movies released after the training date by popularity
- to ensure most popular movies are in our dataset

In [ ]:
# Load environment variables from .env file
load_dotenv()

# Get TMDB API key from environment variable
TMDB_API_KEY = os.getenv("TMDB_API_KEY")
if not TMDB_API_KEY:
    raise ValueError("TMDB_API_KEY not found in environment variables. Please create a .env file with your API key.")

BASE_URL = "https://api.themoviedb.org/3"

START_DATE = "2025-01-01"
END_DATE = "2025-12-31"

def fetch_tmdb_movies_pages(start_date, end_date, start_page=1, end_page=50, sort_by="popularity.desc"):
    """
    Fetch movies from TMDB from a custom page range.
    start_page: first page to fetch
    end_page: last page to fetch (inclusive)
    """
    items = []

    print(f"Fetching movies from page {start_page} to {end_page}...")
    print(f"Date range: {start_date} to {end_date}")
    print(f"Sort: {sort_by}\n")

    for page in tqdm(range(start_page, end_page + 1), desc="Fetching pages"):
        url = f"{BASE_URL}/discover/movie"
        params = {
            "api_key": TMDB_API_KEY,
            "language": "en-US",
            "sort_by": sort_by,
            "page": page,
            "release_date.gte": start_date,
            "release_date.lte": end_date
        }

        try:
            r = requests.get(url, params=params)
            r.raise_for_status()
            data = r.json()

            if not data.get("results"):
                print(f"\nNo more results at page {page}")
                break

            for movie in data["results"]:
                items.append({
                    "title": movie.get("title", ""),
                    "year": (movie.get("release_date") or "")[:4],
                    "release_date": movie.get("release_date", ""),
                    "media_type": "movie",
                    "tmdb_id": movie["id"]
                })

            time.sleep(0.25)

        except requests.exceptions.HTTPError as e:
            if e.response.status_code == 429:
                print(f"\n⚠️ Rate limited at page {page}. Waiting 10 seconds...")
                time.sleep(10)
                continue
            else:
                print(f"\n⚠️ Error at page {page}: {e}")
                break

    return items


# Fetch pages 51 to 100
items = fetch_tmdb_movies_pages(START_DATE, END_DATE, start_page=51, end_page=100, sort_by="popularity.desc")
print(f"\n✓ Fetched {len(items)} movies")

Fetching movies from page 51 to 100...
Date range: 2025-01-01 to 2025-12-31
Sort: popularity.desc



Fetching pages:  46%|████▌     | 23/50 [00:09<00:10,  2.54it/s]


KeyboardInterrupt: 

### Use Wikipedia API to fetch movie plot

In [7]:
# Define Wikipedia API Functions
WIKI_API = "https://en.wikipedia.org/w/api.php"

# Wikipedia requires a User-Agent header to identify your application
# This is required to avoid 403 errors
HEADERS = {
    "User-Agent": "WikipediaPlotExtractor/1.0 (https://example.com/contact; your-email@example.com)"
}


def search_wikipedia(title, year=None, media_type=None):
    """Search for a Wikipedia page by title"""
    query = title
    if media_type == "movie":
        query += " film"
    elif media_type == "tv":
        query += " TV series"

    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json"
    }

    try:
        r = requests.get(WIKI_API, params=params, headers=HEADERS)
        r.raise_for_status()
        results = r.json()["query"]["search"]
        time.sleep(0.5)  # Be respectful to Wikipedia's servers
        return results[0]["title"] if results else None
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 403:
            print(f"⚠️  403 Forbidden - Wikipedia may be blocking requests. Try again later.")
        raise


def get_wikipedia_plot(page_title, debug=False):
    """
    Extract plot/synopsis section from Wikipedia page.
    Tries multiple section name variations.
    """
    params = {
        "action": "query",
        "prop": "revisions",
        "rvprop": "content",
        "rvslots": "main",
        "titles": page_title,
        "format": "json"
    }

    try:
        r = requests.get(WIKI_API, params=params, headers=HEADERS)
        r.raise_for_status()
        pages = r.json()["query"]["pages"]
        time.sleep(0.5)  # Be respectful to Wikipedia's servers
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 403:
            if debug:
                print(f"⚠️  403 Forbidden - Wikipedia may be blocking requests.")
        raise

    page = next(iter(pages.values()))
    if "revisions" not in page:
        if debug:
            print(f"✗ No revisions found for page: {page_title}")
        return None

    text = page["revisions"][0]["slots"]["main"]["*"]
    wikicode = mwparserfromhell.parse(text)
    
    # List of possible plot section names (case-insensitive)
    plot_section_names = [
        "Plot",
        "Plot summary",
        "Synopsis",
        "Story",
        "Premise",
        "Plot and story",
        "Summary"
    ]
    
    # First, try to get all sections and see what's available
    all_sections = wikicode.get_sections(levels=[2])  # Level 2 headings (== Heading ==)
    
    if debug:
        print(f"\nAvailable sections on '{page_title}':")
        for section in all_sections[:10]:  # Show first 10 sections
            # Extract section title
            section_text = str(section)
            match = re.search(r'^==\s*(.+?)\s*==', section_text, re.MULTILINE)
            if match:
                print(f"  - {match.group(1)}")
    
    # Try each plot section name variation
    for section_name in plot_section_names:
        # Try exact match - matches parameter expects a string or callable, not a list
        sections = wikicode.get_sections(matches=section_name)
        if sections:
            plot_text = sections[0].strip_code().strip()
            if plot_text and len(plot_text) > 100:  # Ensure substantial content
                if debug:
                    print(f"✓ Found plot in section: '{section_name}'")
                return plot_text
    
    # Try case-insensitive match by checking all sections
    for section in all_sections:
        section_text = str(section)
        # Extract section heading
        match = re.search(r'^==\s*(.+?)\s*==', section_text, re.MULTILINE | re.IGNORECASE)
        if match:
            heading = match.group(1).strip()
            # Check if heading contains our plot keywords
            if any(keyword.lower() in heading.lower() for keyword in ["plot", "synopsis", "story", "premise"]):
                plot_text = section.strip_code().strip()
                if plot_text and len(plot_text) > 100:
                    if debug:
                        print(f"✓ Found plot in section: '{heading}'")
                    return plot_text
    
    if debug:
        print(f"✗ No plot section found. Tried: {', '.join(plot_section_names)}")
    
    return None


def get_wikipedia_full_page(page_title, debug=False):
    """
    Extract the full Wikipedia page content (all text, cleaned).
    This is useful for RAG projects where you need the entire page content.
    
    Returns:
    - dict with 'full_text' (cleaned text), 'sections' (dict of section_name: content), 
      'introduction' (intro paragraph), 'section_count', and 'total_length'
    """
    params = {
        "action": "query",
        "prop": "revisions",
        "rvprop": "content",
        "rvslots": "main",
        "titles": page_title,
        "format": "json"
    }

    try:
        r = requests.get(WIKI_API, params=params, headers=HEADERS)
        r.raise_for_status()
        pages = r.json()["query"]["pages"]
        time.sleep(0.5)  # Be respectful to Wikipedia's servers
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 403:
            if debug:
                print(f"⚠️  403 Forbidden - Wikipedia may be blocking requests.")
        raise

    page = next(iter(pages.values()))
    if "revisions" not in page:
        if debug:
            print(f"✗ No revisions found for page: {page_title}")
        return None

    text = page["revisions"][0]["slots"]["main"]["*"]
    wikicode = mwparserfromhell.parse(text)
    
    # Get all sections (level 2 headings)
    all_sections = wikicode.get_sections(levels=[2])
    
    # Extract sections with their titles
    sections_dict = {}
    for section in all_sections:
        section_text = str(section)
        # Extract section heading
        match = re.search(r'^==\s*(.+?)\s*==', section_text, re.MULTILINE)
        if match:
            heading = match.group(1).strip()
            # Clean the section content (remove wiki markup)
            section_content = section.strip_code().strip()
            if section_content:
                sections_dict[heading] = section_content
    
    # Get full page text (all content cleaned)
    full_text = wikicode.strip_code().strip()
    
    # Also get the introduction (content before first section)
    intro_match = re.search(r'^(.*?)(?=^==)', str(wikicode), re.MULTILINE | re.DOTALL)
    introduction = ""
    if intro_match:
        intro_wikicode = mwparserfromhell.parse(intro_match.group(1))
        introduction = intro_wikicode.strip_code().strip()
    
    result = {
        "full_text": full_text,
        "introduction": introduction,
        "sections": sections_dict,
        "section_count": len(sections_dict),
        "total_length": len(full_text)
    }
    
    if debug:
        print(f"✓ Extracted full page: {len(full_text)} characters, {len(sections_dict)} sections")
        print(f"  Sections: {', '.join(list(sections_dict.keys())[:5])}")
        if len(sections_dict) > 5:
            print(f"  ... and {len(sections_dict) - 5} more")
    
    return result

print("✓ Wikipedia functions loaded")

✓ Wikipedia functions loaded


In [ ]:
# Configuration - EXTRACTING PLOTS ONLY
OUTPUT_FILE = "../raw_data/movie_raw/tmdb_wikipedia_plots.json"
CHECKPOINT_FILE = "../raw_data/movie_raw/wikipedia_plot_extraction_checkpoint.json"

# ensures raw_data/ and movie_raw/ both exist
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

MIN_PLOT_LENGTH = 300  # Minimum plot length to save
REQUEST_DELAY = 1.5  # Delay between requests (seconds) - increased to avoid blocking
BATCH_SAVE_INTERVAL = 10  # Save checkpoint every N items

def load_checkpoint():
    """Load progress from checkpoint file"""
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
                return json.load(f)
        except:
            return {"processed_indices": [], "documents": []}
    return {"processed_indices": [], "documents": []}

def save_checkpoint(processed_indices, documents):
    """Save progress to checkpoint file"""
    checkpoint = {
        "processed_indices": processed_indices,
        "documents": documents,
        "timestamp": datetime.now().isoformat(),
        "total_processed": len(processed_indices),
        "total_with_plots": len([d for d in documents if d.get("plot")])
    }
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(checkpoint, f, indent=2, ensure_ascii=False)

def save_final_output(documents):
    """Save final results to output file"""
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(documents, f, indent=2, ensure_ascii=False)

# Load checkpoint
checkpoint = load_checkpoint()
existing_documents = checkpoint.get("documents", [])

# Get set of already processed tmdb_ids from checkpoint
processed_tmdb_ids = set()
for doc in existing_documents:
    if doc.get("tmdb_id"):
        processed_tmdb_ids.add(doc["tmdb_id"])

print("="*80)
print("EXTRACTING PLOTS FROM WIKIPEDIA FOR NEW MOVIES")
print("="*80)
print(f"Total movies fetched from TMDB: {len(items)}")
print(f"Already processed (from checkpoint): {len(processed_tmdb_ids)}")
print(f"Items with plots in checkpoint: {len([d for d in existing_documents if d.get('has_plot')])}")

# Filter items to only process movies NOT in checkpoint
items_to_process = [item for item in items if item.get("tmdb_id") not in processed_tmdb_ids]

print(f"New movies to process: {len(items_to_process)}")
print(f"Request delay: {REQUEST_DELAY} seconds (to avoid blocking)")
print("="*80)

if not items_to_process:
    print("\n✓ All movies from TMDB are already in the checkpoint!")
    print("  No new movies to process.")
else:
    print("\n⚠️  IMPORTANT: This will take a while!")
    print(f"   Estimated time: ~{len(items_to_process) * REQUEST_DELAY / 60:.1f} minutes")
    print("   Progress is saved automatically - you can stop and resume anytime")
    print("   Extracting PLOT sections only from Wikipedia\n")

# Start with existing documents
documents = existing_documents.copy()
processed_indices = set(checkpoint.get("processed_indices", []))

if items_to_process:
    # Get the next index to use (continue from where checkpoint left off)
    next_index = max(processed_indices) + 1 if processed_indices else 0
    
    for idx, item in enumerate(tqdm(items_to_process, desc="Processing"), 1):
        i = next_index + idx - 1  # Use sequential indices
        try:
            # Search for Wikipedia page
            page_title = search_wikipedia(
                item["title"],
                item.get("year"),
                item.get("media_type")
            )
            
            wikipedia_title = None
            plot = None
            
            if page_title:
                wikipedia_title = page_title
                # Get PLOT from Wikipedia (not full page)
                plot = get_wikipedia_plot(page_title, debug=False)
            
            # Create document with all metadata and plot
            document = {
                # TMDB metadata
                "title": item["title"],
                "year": item.get("year", ""),
                "media_type": item.get("media_type", ""),
                "tmdb_id": item.get("tmdb_id"),
                
                # Wikipedia metadata
                "wikipedia_title": wikipedia_title,
                "plot": plot if plot and len(plot) >= MIN_PLOT_LENGTH else None,
                "plot_length": len(plot) if plot else 0,
                "has_plot": plot is not None and len(plot) >= MIN_PLOT_LENGTH
            }
            
            documents.append(document)
            processed_indices.add(i)
            processed_tmdb_ids.add(item.get("tmdb_id"))  # Track by tmdb_id too
            
            # Save checkpoint periodically
            if idx % BATCH_SAVE_INTERVAL == 0:
                save_checkpoint(list(processed_indices), documents)
                print(f"\n  ✓ Checkpoint saved: {idx}/{len(items_to_process)} new movies processed")
            
        except requests.exceptions.HTTPError as e:
            if e.response.status_code == 403:
                print(f"\n⚠️  403 Forbidden detected! Pausing for 60 seconds...")
                print(f"   Processed {idx}/{len(items_to_process)} so far")
                save_checkpoint(list(processed_indices), documents)
                time.sleep(60)  # Wait 1 minute before continuing
                continue
            else:
                # Other HTTP errors - skip this item
                document = {
                    "title": item["title"],
                    "year": item.get("year", ""),
                    "media_type": item.get("media_type", ""),
                    "tmdb_id": item.get("tmdb_id"),
                    "wikipedia_title": None,
                    "plot": None,
                    "plot_length": 0,
                    "has_plot": False,
                    "error": f"HTTP {e.response.status_code}"
                }
                documents.append(document)
                processed_indices.add(i)
                processed_tmdb_ids.add(item.get("tmdb_id"))
        except Exception as e:
            # Other errors - skip this item but record it
            document = {
                "title": item["title"],
                "year": item.get("year", ""),
                "media_type": item.get("media_type", ""),
                "tmdb_id": item.get("tmdb_id"),
                "wikipedia_title": None,
                "plot": None,
                "plot_length": 0,
                "has_plot": False,
                "error": str(e)[:100]  # Truncate long error messages
            }
            documents.append(document)
            processed_indices.add(i)
            processed_tmdb_ids.add(item.get("tmdb_id"))
        
        # Rate limiting delay
        time.sleep(REQUEST_DELAY)

# Final save
save_checkpoint(list(processed_indices), documents)
save_final_output(documents)

# Print summary
print("\n" + "="*80)
print("EXTRACTION COMPLETE")
print("="*80)
print(f"Total items processed: {len(documents)}")
print(f"Items with plots found: {len([d for d in documents if d.get('has_plot')])}")
print(f"Items without plots: {len([d for d in documents if not d.get('has_plot')])}")
print(f"\nResults saved to: {OUTPUT_FILE}")
print(f"Checkpoint saved to: {CHECKPOINT_FILE}")
print("="*80)

# Show some statistics
if documents:
    items_with_plots = [d for d in documents if d.get('has_plot')]
    if items_with_plots:
        avg_plot_length = sum(d.get('plot_length', 0) for d in items_with_plots) / len(items_with_plots)
        print(f"\nAverage plot length: {avg_plot_length:.0f} characters")
        
        # Show example
        example = next((d for d in items_with_plots if d.get('plot')), None)
        if example:
            print(f"\nExample (from '{example['title']}'):")
            print(f"  - Plot length: {example.get('plot_length', 0)} characters")
            plot_preview = example.get('plot', '')[:200]
            if plot_preview:
                print(f"  - Plot preview: {plot_preview}...")
        
        print(f"\n💡 Each document contains: title, year, tmdb_id, wikipedia_title, plot, plot_length")

EXTRACTING PLOTS FROM WIKIPEDIA FOR NEW MOVIES
Total movies fetched from TMDB: 999
Already processed (from checkpoint): 1240
Items with plots in checkpoint: 921
New movies to process: 886
Request delay: 1.5 seconds (to avoid blocking)

⚠️  IMPORTANT: This will take a while!
   Estimated time: ~22.1 minutes
   Progress is saved automatically - you can stop and resume anytime
   Extracting PLOT sections only from Wikipedia



Processing:   1%|          | 9/886 [00:28<48:18,  3.30s/it]


  ✓ Checkpoint saved: 10/886 new movies processed


Processing:   2%|▏         | 19/886 [00:58<44:16,  3.06s/it]


  ✓ Checkpoint saved: 20/886 new movies processed


...


  ✓ Checkpoint saved: 40/886 new movies processed


Processing:   6%|▌         | 49/886 [02:26<42:49,  3.07s/it]


  ✓ Checkpoint saved: 50/886 new movies processed


Processing:   6%|▋         | 57/886 [02:52<41:41,  3.02s/it]


KeyboardInterrupt: 

#### Clean fetch data
- remove entries without plots and mis-extracted entries
- filter out entries with year mismatch
- filter out movies with release date before training date

In [ ]:
# Clean checkpoint data: Filter out entries without plots and mis-extracted entries
CHECKPOINT_FILE = "../raw_data/movie_raw/wikipedia_plot_extraction_checkpoint.json"
CLEANED_OUTPUT_FILE = "../raw_data/movie_raw/tmdb_wikipedia_plots_cleaned.json"

print("="*80)
print("CLEANING CHECKPOINT DATA")
print("="*80)

# Load checkpoint
with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
    checkpoint = json.load(f)

all_documents = checkpoint.get("documents", [])
print(f"Total documents in checkpoint: {len(all_documents)}")

# Filter criteria:
# 1. has_plot must be True
# 2. plot must not be None/empty
# 3. plot_length must be >= 300 (minimum length)
# 4. Filter out mis-extracted entries (e.g., "List of..." pages, wrong Wikipedia titles)

def normalize_title(title):
    """Normalize title by removing years, (film), (movie) markers"""
    if not title:
        return ""
    # Remove (YYYY) or (YYYY film) patterns
    title = re.sub(r'\s*\(\d{4}.*?\)', '', title)
    # Remove standalone (film) or (movie)
    title = re.sub(r'\s*\(film\)', '', title, flags=re.IGNORECASE)
    title = re.sub(r'\s*\(movie\)', '', title, flags=re.IGNORECASE)
    return title.strip().lower()

def is_valid_entry(doc):
    """Check if document has valid plot extraction"""
    # Must have plot
    if not doc.get("has_plot"):
        return False
    
    plot = doc.get("plot")
    if not plot or len(plot.strip()) == 0:
        return False
    
    # Minimum plot length
    if doc.get("plot_length", 0) < 300:
        return False
    
    # Filter out mis-extracted entries
    wikipedia_title = doc.get("wikipedia_title", "")
    movie_title = doc.get("title", "")
    movie_year = doc.get("year", "")
    
    if not wikipedia_title:
        return False
    
    # Skip "List of..." pages (these are usually wrong)
    if wikipedia_title.lower().startswith("list of"):
        return False
    
    # Check for year mismatch - extract year from Wikipedia title
    wiki_year_match = re.search(r'\((\d{4})', wikipedia_title)
    if wiki_year_match and movie_year:
        wiki_year = wiki_year_match.group(1)
        # If years differ significantly (more than 2 years), likely wrong
        try:
            year_diff = abs(int(movie_year) - int(wiki_year))
            if year_diff > 2:
                return False
        except:
            pass
    
    # Normalize titles for comparison (remove years and film markers)
    norm_movie = normalize_title(movie_title)
    norm_wiki = normalize_title(wikipedia_title)
    
    # Extract meaningful words (excluding common words)
    movie_words = set(norm_movie.split())
    wiki_words = set(norm_wiki.split())
    common_words = {"the", "a", "an", "of", "and", "or"}
    movie_words = movie_words - common_words
    wiki_words = wiki_words - common_words
    
    # Remove any remaining digits
    movie_words = {w for w in movie_words if not w.isdigit()}
    wiki_words = {w for w in wiki_words if not w.isdigit()}
    
    if not movie_words:
        return False
    
    # Check if all movie words are in wiki words
    if not (movie_words <= wiki_words):
        return False
    
    # For single-word titles, check if wiki has significant additional words
    # (e.g., "Shadows" vs "Dark Shadows" - "dark" is additional, so reject)
    if len(movie_words) == 1:
        additional_words = wiki_words - movie_words
        # If there are additional words beyond common ones, likely wrong
        if additional_words:
            return False
    
    # For 2-word titles, allow 1 additional word max
    elif len(movie_words) == 2:
        additional_words = wiki_words - movie_words
        if len(additional_words) > 1:
            return False
    
    # For longer titles (3+ words), use percentage matching
    else:
        matching_words = movie_words & wiki_words
        if len(matching_words) / len(movie_words) < 0.8:
            return False
    
    return True

# Filter documents
cleaned_documents = [doc for doc in all_documents if is_valid_entry(doc)]

print(f"\nFiltering results:")
print(f"  - Total documents: {len(all_documents)}")
print(f"  - Documents with has_plot=True: {sum(1 for d in all_documents if d.get('has_plot'))}")
print(f"  - Documents with plot: {sum(1 for d in all_documents if d.get('plot'))}")
print(f"  - Documents with plot_length >= 300: {sum(1 for d in all_documents if d.get('plot_length', 0) >= 300)}")
print(f"  - Valid cleaned documents: {len(cleaned_documents)}")
print(f"  - Removed: {len(all_documents) - len(cleaned_documents)}")

# Show some examples of removed entries
removed = [d for d in all_documents if not is_valid_entry(d)]
if removed:
    print(f"\n{'='*80}")
    print("SAMPLE OF REMOVED ENTRIES:")
    print("="*80)
    for i, doc in enumerate(removed[:5], 1):
        print(f"\n{i}. {doc.get('title')} ({doc.get('year')})")
        print(f"   Wikipedia: {doc.get('wikipedia_title', 'N/A')}")
        print(f"   has_plot: {doc.get('has_plot')}")
        print(f"   plot_length: {doc.get('plot_length', 0)}")
        if doc.get('wikipedia_title', '').lower().startswith('list of'):
            print(f"   Reason: Wikipedia title is a 'List of...' page (mis-extracted)")

# Save cleaned data
with open(CLEANED_OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(cleaned_documents, f, indent=2, ensure_ascii=False)

print(f"\n{'='*80}")
print("CLEANING COMPLETE")
print("="*80)
print(f"✓ Cleaned data saved to: {CLEANED_OUTPUT_FILE}")
print(f"  Total valid documents: {len(cleaned_documents)}")
print(f"  Average plot length: {sum(d.get('plot_length', 0) for d in cleaned_documents) / len(cleaned_documents) if cleaned_documents else 0:.0f} characters")
print("="*80)

CLEANING CHECKPOINT DATA
Total documents in checkpoint: 1240

Filtering results:
  - Total documents: 1240
  - Documents with has_plot=True: 921
  - Documents with plot: 921
  - Documents with plot_length >= 300: 921
  - Valid cleaned documents: 644
  - Removed: 596

SAMPLE OF REMOVED ENTRIES:

1. Strangers (2024)
   Wikipedia: The Strangers (2008 film)
   has_plot: True
   plot_length: 2622

2. Murder at the Embassy (2025)
   Wikipedia: Murder at the Embassy
   has_plot: False
   plot_length: 235

3. War of the Worlds (2025)
   Wikipedia: List of works based on The War of the Worlds
   has_plot: False
   plot_length: 0
   Reason: Wikipedia title is a 'List of...' page (mis-extracted)

4. Icefall (2025)
   Wikipedia: Icefall (film)
   has_plot: False
   plot_length: 152

5. Demon Slayer: Kimetsu no Yaiba Infinity Castle (2025)
   Wikipedia: List of Demon Slayer: Kimetsu no Yaiba episodes
   has_plot: False
   plot_length: 0
   Reason: Wikipedia title is a 'List of...' page (mis-extract

#### Optional: Add more movies using wikipedia api 
- are some less popular movies to expand the scope of dataset

In [ ]:
WIKI_API = "https://en.wikipedia.org/w/api.php"
HEADERS = {
    "User-Agent": "RAGMovieBot/1.0 (contact: your_email@domain.com)"
}

def get_movies_from_category(category, max_pages=500):
    movies = []
    cmcontinue = None

    while True:
        params = {
            "action": "query",
            "list": "categorymembers",
            "cmtitle": f"Category:{category}",
            "cmtype": "page",
            "cmlimit": "500",
            "format": "json",
        }

        if cmcontinue:
            params["cmcontinue"] = cmcontinue

        r = requests.get(WIKI_API, params=params, headers=HEADERS)
        r.raise_for_status()
        data = r.json()

        pages = data["query"]["categorymembers"]
        movies.extend(pages)

        if len(movies) >= max_pages:
            break

        cmcontinue = data.get("continue", {}).get("cmcontinue")
        if not cmcontinue:
            break

    return movies[:max_pages]

In [ ]:
# fetch movies from 2024, 2025, and 2026 categories
years = [2024, 2025, 2026]
all_movies = []

for year in years:
    category = f"{year} films"
    movies = get_movies_from_category(category)
    print(f"{year}: {len(movies)} movies")
    all_movies.extend(movies)

2024: 500 movies
2025: 500 movies
2026: 432 movies


In [ ]:
# Check which movies from all_movies are missing from movie_chunks and fetch their plots
print("\nChecking for missing movies in movie_chunks...")
print("=" * 80)

# Load movie_chunks to see which Wikipedia titles already have chunks
try:
    with open("../chunk_data/movie_chunks.json", "r", encoding="utf-8") as f:
        existing_chunks = json.load(f)
    
    # Extract wikipedia titles from existing chunks
    existing_wikipedia_titles = set()
    for chunk in existing_chunks:
        wiki_title = chunk.get("metadata", {}).get("wikipedia_id")
        if wiki_title:
            existing_wikipedia_titles.add(wiki_title)
    
    print(f"Movies already in chunks: {len(existing_wikipedia_titles)}")
except FileNotFoundError:
    print("⚠️  movie_chunks.json not found. All movies will be considered missing.")
    existing_wikipedia_titles = set()

# Find movies that don't have chunks (by Wikipedia title)
missing_movies = []
for movie in all_movies:
    wiki_title = movie.get("title", "")
    if wiki_title and wiki_title not in existing_wikipedia_titles:
        missing_movies.append(movie)

print(f"Movies missing from chunks: {len(missing_movies)}")
print("=" * 80)

# If there are missing movies, fetch their plots from Wikipedia
if missing_movies:
    print(f"\nFetching plots for {len(missing_movies)} missing movies...")
    
    # Fetch plots for missing movies
    MIN_PLOT_LENGTH = 300
    new_movies_with_plots = []
    skipped_no_plot = 0
    skipped_error = 0
    BATCH_SAVE_INTERVAL = 50  # Save status every N movies
    CLEANED_FILE = "../raw_data/movie_raw/tmdb_wikipedia_plots_cleaned.json"
    
    # Load existing cleaned data at the start
    try:
        with open(CLEANED_FILE, "r", encoding="utf-8") as f:
            existing_movies = json.load(f)
        print(f"  ✓ Loaded {len(existing_movies)} existing movies from cleaned file")
    except FileNotFoundError:
        existing_movies = []
        print(f"  ℹ️  No existing file found, will create new one")
    
    print(f"  Minimum plot length: {MIN_PLOT_LENGTH} characters")
    print(f"  Progress will be saved every {BATCH_SAVE_INTERVAL} movies\n")
    
    for idx, movie in enumerate(tqdm(missing_movies, desc="Fetching plots"), 1):
        try:
            wiki_title = movie.get("title", "")
            if not wiki_title:
                skipped_no_plot += 1
                continue
            
            # Get plot from Wikipedia (using existing get_wikipedia_plot function)
            plot = get_wikipedia_plot(wiki_title, debug=False)
            
            # Only add if we got a plot (skip if no plot)
            if plot and len(plot) >= MIN_PLOT_LENGTH:
                # Extract year from title if possible (e.g., "Movie (2024 film)")
                year_match = re.search(r'\((\d{4})', wiki_title)
                year = year_match.group(1) if year_match else ""
                
                # Extract base title (remove year and "(film)" markers)
                base_title = re.sub(r'\s*\(\d{4}.*?\)', '', wiki_title).strip()
                base_title = re.sub(r'\s*\(film\)', '', base_title, flags=re.IGNORECASE).strip()
                
                new_movie = {
                    "title": base_title,
                    "year": year,
                    "media_type": "movie",
                    "tmdb_id": None,  # We don't have TMDB ID for Wikipedia-only movies
                    "wikipedia_title": wiki_title,
                    "plot": plot,
                    "plot_length": len(plot),
                    "has_plot": True
                }
                new_movies_with_plots.append(new_movie)
            else:
                skipped_no_plot += 1
        
        except Exception as e:
            skipped_error += 1
            print(f"\n⚠️  Error fetching plot for '{movie.get('title', 'unknown')}': {e}")
            continue
        
        # Periodic status update and save
        if idx % BATCH_SAVE_INTERVAL == 0:
            # Save progress to JSON file
            all_movies_to_save = existing_movies + new_movies_with_plots
            with open(CLEANED_FILE, "w", encoding="utf-8") as f:
                json.dump(all_movies_to_save, f, indent=2, ensure_ascii=False)
            
            print(f"\n  📊 Status update ({idx}/{len(missing_movies)}):")
            print(f"     ✓ Movies with plots: {len(new_movies_with_plots)}")
            print(f"     ✗ Skipped (no plot): {skipped_no_plot}")
            print(f"     ⚠️  Errors: {skipped_error}")
            print(f"     💾 Progress saved to JSON ({len(all_movies_to_save)} total movies)")
    
    # Final status summary
    print(f"\n{'='*80}")
    print("FETCHING COMPLETE - SUMMARY")
    print("="*80)
    print(f"Total movies processed: {len(missing_movies)}")
    print(f"  ✓ Movies with plots found: {len(new_movies_with_plots)}")
    print(f"  ✗ Skipped (no plot): {skipped_no_plot}")
    print(f"  ⚠️  Errors: {skipped_error}")
    print("="*80)
    
    # Final save - ensure all new movies are saved
    if new_movies_with_plots:
        # Combine existing and new movies
        all_movies_to_save = existing_movies + new_movies_with_plots
        
        # Save final data
        with open(CLEANED_FILE, "w", encoding="utf-8") as f:
            json.dump(all_movies_to_save, f, indent=2, ensure_ascii=False)
        
        print(f"\n{'='*80}")
        print("SAVE COMPLETE")
        print("="*80)
        print(f"✓ Added {len(new_movies_with_plots)} new movies with plots")
        print(f"✓ Total movies in cleaned file: {len(all_movies_to_save)}")
        print(f"  (Previously had: {len(existing_movies)})")
        print("="*80)
    else:
        print(f"\n⚠️  No new plots fetched (all missing movies were skipped)")
else:
    print("\n✓ All movies already have chunks!")


Checking for missing movies in movie_chunks...
Movies already in chunks: 326
Movies missing from chunks: 1370

Fetching plots for 1370 missing movies...
  ℹ️  No existing file found, will create new one
  Minimum plot length: 300 characters
  Progress will be saved every 50 movies



Fetching plots:   4%|▎         | 50/1370 [00:35<15:24,  1.43it/s]


  📊 Status update (50/1370):
     ✓ Movies with plots: 29
     ✗ Skipped (no plot): 21
     ⚠️  Errors: 0
     💾 Progress saved to JSON (29 total movies)


Fetching plots:   7%|▋         | 100/1370 [01:10<14:51,  1.42it/s]


  📊 Status update (100/1370):
     ✓ Movies with plots: 57
     ✗ Skipped (no plot): 43
     ⚠️  Errors: 0
     💾 Progress saved to JSON (57 total movies)


...


  📊 Status update (1300/1370):
     ✓ Movies with plots: 582
     ✗ Skipped (no plot): 718
     ⚠️  Errors: 0
     💾 Progress saved to JSON (582 total movies)


Fetching plots:  99%|█████████▊| 1350/1370 [16:02<00:14,  1.41it/s]


  📊 Status update (1350/1370):
     ✓ Movies with plots: 591
     ✗ Skipped (no plot): 759
     ⚠️  Errors: 0
     💾 Progress saved to JSON (591 total movies)


Fetching plots: 100%|██████████| 1370/1370 [16:16<00:00,  1.40it/s]


FETCHING COMPLETE - SUMMARY
Total movies processed: 1370
  ✓ Movies with plots found: 595
  ✗ Skipped (no plot): 775
  ⚠️  Errors: 0

SAVE COMPLETE
✓ Added 595 new movies with plots
✓ Total movies in cleaned file: 595
  (Previously had: 0)
